In [ ]:
import polars as  pl

In [ ]:
base = pl.scan_parquet("new_base.parquet")

base.head(3).collect()

In [ ]:
"""
Useful stats on content length
"""
(
    base.select(
        pl.col("content").str.len_chars().min().alias("min_article_len"),
        pl.col("content").str.len_chars().max().alias("max_article_len"),
        pl.col("content").str.len_chars().mean().alias("avg_article_len"),
        pl.col("content").str.len_chars().median().alias("median_article_len")
    )
    .collect()
)

# Data cleaning

In [ ]:
# extract topic names from topics struct
chunks = base.with_columns(
    topics=pl.col("topics").list
    .eval(pl.element().struct.field("topic"))
)

# drop the year and quarter_year columns
# only keep published_date for date
chunks = chunks.drop(["year", "quarter_year"])

chunks.collect()

# Phase 3: chunking (lazy)

> chunking runs inside the lazy pipeline: a `map_elements` UDF (python + tiktoken) produces the list of chunks per article, then native `explode` / `int_range` / `pl.format` expressions build the chunk rows.
> the UDF is opaque to the query optimiser, but the plan stays lazy end-to-end so `sink_parquet` still works.
>
> the paragraph handling from `src/preprocessing/token_chunking.py` is removed here: the dataset is verified newline-free (content is a single line per article), so it was dead code. `src/` keeps the general version.
>
> chunk sizes are retargeted from the src/ defaults (700/900/100 → 350/500/75): articles are short (median ~980 tokens, 41% fit in one 900-token chunk), and medium chunks embed more sharply for figure-extraction queries.


In [ ]:
%env TIKTOKEN_CACHE_DIR=/home/sagemaker-user/.cache/tiktoken

In [ ]:
import os
import re
from typing import Sequence

import tiktoken

# sentence-aware chunking, adapted from src/preprocessing/token_chunking.py
# kept self-contained so the notebook runs standalone
#
# note: the paragraph splitting on "\n\n" and None/NaN coercion from src/ are
# dropped here — the source dataset stores each article as a single non-null
# line (verified), so they were dead code. whitespace runs are collapsed to a
# single space inline. a guard assertion in the test cell below fails loudly
# if newlines (paragraph structure) ever appear in content.

EMBEDDING_MODEL = "text-embedding-3-small"

SENTENCE_BOUNDARY_PATTERN = re.compile(r"(?<=[.!?])\s+(?=[\"'(\[]?[A-Z0-9])")

assert os.path.exists(
    os.path.join(
        os.environ["TIKTOKEN_CACHE_DIR"], 
        "9b5ad71b2ce5302211f9c61530b329a4922fc6a4"
    )
), "tiktoken cache dir not found, please set TIKTOKEN_CACHE_DIR to the correct path"

def get_token_encoder(model: str = EMBEDDING_MODEL) -> tiktoken.Encoding:
    """Return the tiktoken encoder for the embedding model."""
    try:
        return tiktoken.encoding_for_model(model)
    except KeyError:
        return tiktoken.get_encoding("cl100k_base")


def count_tokens(text: str, encoder: tiktoken.Encoding) -> int:
    """Count tokens using the configured tiktoken encoder."""
    return len(encoder.encode(text))


def split_into_sentences(text: str) -> list[str]:
    """Split article text into sentence-like units.

    Collapses any whitespace run to a single space. The source dataset stores
    each article as a single line, so no paragraph handling is needed.
    """
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    
    return [
        piece.strip() 
        for piece in SENTENCE_BOUNDARY_PATTERN.split(text) 
        if piece.strip()
        ]


def split_long_unit_by_tokens(
    text: str,
    max_tokens: int,
    encoder: tiktoken.Encoding,
) -> list[str]:
    """
    Split long sentence-like units only when it cannot fit in a chunk.

    Hard fallback to token-based splitting if a single sentence exceeds the max chunk size.
    """
    # encode the text to get the token count
    tokens = encoder.encode(text)
    if len(tokens) <= max_tokens:
        return [text]

    parts = []
    # split the tokens into chunks of size max_tokens
    for start in range(0, len(tokens), max_tokens):
        part = encoder.decode(tokens[start : start + max_tokens]).strip()
        if part:
            parts.append(part)

    return parts


def build_overlap_units(
    units: Sequence[str],
    overlap_tokens: int,
    encoder: tiktoken.Encoding,
) -> list[str]:
    """
    Build an overlap from the end of the previous chunk on unit boundaries.
    """
    if overlap_tokens <= 0:
        return []

    overlap: list[str] = []
    total_tokens = 0

    for unit in reversed(units):
        unit_tokens = count_tokens(unit, encoder)
        if unit_tokens > overlap_tokens:
            break
        if total_tokens + unit_tokens > overlap_tokens:
            break
        overlap.insert(0, unit)
        total_tokens += unit_tokens

    return overlap


def chunk_content_by_tokens(
    content: str,
    encoder: tiktoken.Encoding,
    target_chunk_tokens: int = 700,
    max_chunk_tokens: int = 900,
    overlap_tokens: int = 100,
) -> list[str]:
    """
    Split article content into token-aware, sentence-aware chunks.

    The function prefers sentence boundaries, groups sentences around the target
    size, and uses sentence-aware overlap between chunks.
    """
    if target_chunk_tokens <= 0:
        raise ValueError("target_chunk_tokens must be greater than zero")
    if max_chunk_tokens <= 0:
        raise ValueError("max_chunk_tokens must be greater than zero")
    if target_chunk_tokens > max_chunk_tokens:
        raise ValueError("target_chunk_tokens cannot exceed max_chunk_tokens")
    if overlap_tokens < 0:
        raise ValueError("overlap_tokens cannot be negative")
    if overlap_tokens >= max_chunk_tokens:
        raise ValueError("overlap_tokens must be smaller than max_chunk_tokens")

    raw_units: list[str] = split_into_sentences(content)
    units: list[str] = []
    for unit in raw_units:
        units.extend(split_long_unit_by_tokens(unit, max_chunk_tokens, encoder))

    chunks: list[str] = []
    current_units: list[str] = []
    current_tokens = 0

    for unit in units:
        unit_tokens = count_tokens(unit, encoder)

        # flush the current chunk if adding this unit would exceed the max chunk size
        # or if the current chunk has reached the target size
        should_flush = current_units and (
            current_tokens + unit_tokens > max_chunk_tokens
            or current_tokens >= target_chunk_tokens
        )

        if should_flush:
            # flush the current chunk
            chunks.append(" ".join(current_units).strip())

            # build the overlap for the next chunk
            current_units = build_overlap_units(current_units, overlap_tokens, encoder)
            current_tokens = sum(count_tokens(item, encoder) for item in current_units)

        current_units.append(unit)
        current_tokens += unit_tokens

    # flush any remaining units into a final chunk
    if current_units:
        chunks.append(" ".join(current_units).strip())

    return [chunk for chunk in chunks if chunk]


encoder = get_token_encoder()

In [ ]:
# medium chunks: median article is ~980 tokens, so the src/ defaults (700/900/100)
# made chunks nearly article-sized (41% of articles = 1 chunk) — too coarse for
# figure-extraction queries, which need sharp embeddings.
# overlap 75 keeps zero-overlap boundaries rare (~10%) at ~11% corpus redundancy;
# 100 would halve the former but push duplication to ~17% (near-duplicate top-k hits).
TARGET_CHUNK_TOKENS = 350
MAX_CHUNK_TOKENS = 500
OVERLAP_TOKENS = 75


def build_rag_chunks(lf: pl.LazyFrame) -> pl.LazyFrame:
    # chunk content inside the lazy pipeline
    # (python UDF, opaque to the optimiser but the plan stays lazy)
    lf = (
        lf.with_columns(
            content_chunk=pl.col("content").map_elements(
                lambda content: chunk_content_by_tokens(
                    content,
                    encoder,
                    target_chunk_tokens=TARGET_CHUNK_TOKENS,
                    max_chunk_tokens=MAX_CHUNK_TOKENS,
                    overlap_tokens=OVERLAP_TOKENS,
                ),
                return_dtype=pl.List(pl.String),
            )
        )
        .drop("content")
        .explode("content_chunk")
        # articles with empty content explode to a single null chunk
        .filter(pl.col("content_chunk").is_not_null()
                & (pl.col("content_chunk") != ""))
    )

    # number chunks within each article, then 
    lf = lf.with_columns(
        chunk_index=pl.int_range(pl.len())
        .over("article_id")
    )
    
    # derive chunk_id
    lf = lf.with_columns(
        chunk_id=pl.format("{}:chunk_{}", pl.col("article_id"), pl.col("chunk_index"))
    )

    # retrieval_text used for embedding: title + topics + excerpt
    # (same format as build_retrieval_text_for_rag in the old pipeline)
    lf = lf.with_columns(
        retrieval_text=pl.format(
            "Title: {}\nTopics: {}\n\nArticle excerpt:\n{}",
            pl.col("title"),
            pl.col("topics").list.join(", "),
            pl.col("content_chunk"),
        )
    )

    return lf.select([
        "chunk_id",
        "article_id",
        "chunk_index",
        "content_chunk",
        "retrieval_text",
        "title",
        "published_date",
        "news_site",
        "url",
        "topics",
    ])


In [ ]:
# the simplified chunker collapses whitespace itself, but still assumes:
# no nulls (coercion removed) and no newlines (paragraph-aware splitting removed)
def assert_content_contains_no_nulls(lf: pl.LazyFrame) -> None:
    """Assert that the content column contains no nulls."""
    assert lf.select(pl.col("content").is_null().not_().all()).collect().item(), \
        "content contains nulls"
    
def assert_content_contains_only_single_paragraph(lf: pl.LazyFrame) -> None:
    """Assert that the content column contains only a single paragraph (no newlines)."""
    assert lf.select(pl.col("content").str.contains("\n").not_().all()).collect().item(), \
        "content contains newlines, restore paragraph-aware splitting"

# test that the content column contains no nulls and no newlines (single paragraph)
assert_content_contains_no_nulls(chunks)
assert_content_contains_only_single_paragraph(chunks)

# test chunking on a small subset of articles
sample_chunks = build_rag_chunks(chunks.head(5)).collect()

# every chunk_id must be unique and every chunk non-empty
assert sample_chunks.get_column("chunk_id").n_unique() == len(sample_chunks)
assert (sample_chunks.get_column("content_chunk").str.len_chars() > 0).all()

sample_chunks.head(3)

# Full run

> chunking stays lazy: the UDF executes at sink time. `new_chunks.parquet` is the input for `4.embedding.ipynb`.

In [ ]:
build_rag_chunks(chunks).sink_parquet("new_chunks.parquet")
print("done")